# Query Esri Content for User  
Use `gs-agent` env, for arcgis tools. 

In [5]:
from arcgis.gis import GIS
import pandas as pd
from dotenv import load_dotenv
import os
import json

load_dotenv()

True

Import and Sign-in

In [2]:

signin_name = os.getenv('ESRI_USERNAME')
signin_password = os.getenv('ESRI_PASSWORD')
signin_link = os.getenv("HUBLINK")

# Connect to the organization
gis = GIS(signin_link, signin_name, signin_password )
username = gis.users.me.username

print("Done")

Done


## Get all content from the USER and document size  
Produces a bunch of warnings, ignore them.

In [3]:

# 1. Search for all public feature layers owned by the client's org/account
# brings everything - insert username from gis var
all_my_items = gis.content.search(query= f"owner:{gis.users.me.username}", max_items=1000)
print( all_my_items )

data_summary = []

# iterate through items
for item in all_my_items:
    size_mb = item.size / (1024 * 1024)
    item_type = item.type
    
    # Calculate the credit rate based on Esri's storage rules
    if item_type == "Feature Service":
        # Feature storage: 2.4 credits per 10 MB
        monthly_credits = (size_mb / 10) * 2.4
    elif item_type in ["Imagery Layer", "Map Service", "Vector Tile Service", "Scene Service"]:
        # Raster, Imagery, Tile storage: 1.2 credits per 1 GB
        size_gb = size_mb / 1024
        monthly_credits = size_gb * 1.2
    else:
        # File storage (Files, CSVs, Notebooks, Web Maps): 1.2 credits per 1 GB
        size_gb = size_mb / 1024
        monthly_credits = size_gb * 1.2

    data_summary.append({
        "Item Name": item.title,
        "Type": item_type,
        "Size (MB)": round(size_mb, 2),
        "Est. Credits/Month": round(monthly_credits, 4),
        "Access": item.access
    })

df = pd.DataFrame(data_summary)

df.sort_values( by='Size (MB)' , ascending=False).head(20)

c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#t

[<Item title:"ports_and_transferfacilities" type:GeoJson owner:sara2263_thrive_geohub>, <Item title:"Protected Lands Dataset for the Chattanooga Region" type:GeoJson owner:sara2263_thrive_geohub>, <Item title:"ACS Housing Units Occupancy" type:GeoJson owner:sara2263_thrive_geohub>, <Item title:"map_extents" type:Feature Layer Collection owner:sara2263_thrive_geohub>, <Item title:"basemap_greyscale_v1_042026" type:Vector Tile Layer owner:sara2263_thrive_geohub>, <Item title:"LancoverChangeGrids" type:GeoJson owner:sara2263_thrive_geohub>, <Item title:"LancoverChangeGrids" type:Feature Layer Collection owner:sara2263_thrive_geohub>, <Item title:"landuse " type:Web Map owner:sara2263_thrive_geohub>, <Item title:"exp builder testing" type:Web Map owner:sara2263_thrive_geohub>, <Item title:"Hub Data Submission - Approved Items" type:Feature Layer Collection owner:sara2263_thrive_geohub>, <Item title:"basemap_v2" type:Web Map owner:sara2263_thrive_geohub>, <Item title:"census_tracts_hub_desi

c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#t

,Item Name,Type,Size (MB),Est. Credits/Month,Access
70,ACS Housing Units Occupancy tracts,Feature Service,1220.33,292.8788,public
20,Landscape Conservation Priority Model,GeoJson,490.84,0.5752,private
47,ACS Housing Units Occupancy (Latest),File Geodatabase,397.46,0.4658,private
29,ACS Housing Units Occupancy tracts,File Geodatabase,345.31,0.4047,private
27,Master_Thrive_2020_2021_Parcels_50_Acres_Conse...,GeoJson,328.74,0.3852,private
84,NHD_waterbodies_Thrive_region_simplified,GeoJson,55.80,0.0654,public
71,NHD_waterbodies_Thrive_region_simplified,Feature Service,29.07,6.9769,private
31,trails_county_name_added,GeoJson,18.16,0.0213,private
50,Chattanooga Regional Trails,GeoJson,14.90,0.0175,private
46,basemap_greyscale,Vector Tile Service,13.70,0.0161,public


## Find Zombie Datasets  
These datasets are not used in any tool or applications. This does not mean that they are useless.

In [ ]:

all_my_items = gis.content.search(query=f"owner:{username}", max_items=1000)

# Separate datasets from applications/maps
datasets = []
containers = []

# Filter items into datasets vs things that contain datasets
for item in all_my_items:
    if item.type in ["Feature Service", "Raster Layer", "Imagery Layer"]:
        datasets.append(item)
    elif item.type in ["Web Map", "Web Scene", "Web Experience", "Dashboard", "Operation View"]:
        containers.append(item)

# Extract layer URLs being used inside maps and apps
used_service_urls = set()

for container in containers:
    try:
        # Get the internal JSON configuration data of the map/app
        data = container.get_data()
        if not data:
            continue
            
        # Convert dict to string to search for service paths quickly
        data_str = json.dumps(data)
        
        # Look for references to this organization's services
        for dataset in datasets:
            if dataset.url and dataset.url in data_str:
                used_service_urls.add(dataset.url)
                
    except Exception:
        # Skip if item configuration isn't readable
        continue

# Find the datasets whose URLs were never found inside any container
zombie_datasets = []

for dataset in datasets:
    if dataset.url not in used_service_urls:
        size_mb = dataset.size / (1024 * 1024)
        zombie_datasets.append({
            "Title": dataset.title,
            "ID": dataset.id,
            "Type": dataset.type,
            "Size (MB)": round(size_mb, 2)
        })

# Print results
print(f"\n--- Found {len(zombie_datasets)} Zombie Datasets ---")
for zombie in zombie_datasets:
    print(f"Name: {zombie['Title']} | Type: {zombie['Type']} | Size: {zombie['Size (MB)']} MB | ID: {zombie['ID']}")

c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#t


--- Found 13 Zombie Datasets ---
Name: Hub Data Submission - Approved Items | Type: Feature Service | Size: 0.0 MB | ID: 6a636659b43742bab089cf082798c7b2
Name: census_tracts_hub_design_joined | Type: Feature Service | Size: 0.74 MB | ID: 52f2e545c5374c1f9ead7bb821ac063a
Name: Hub Data Submission - Example_results | Type: Feature Service | Size: 0.0 MB | ID: 2962df8685ff4963a7716c214b31790e
Name: ACS_2024_housing_occupancy_Thrive_county | Type: Feature Service | Size: 3.57 MB | ID: 59e927a076dd4c1e85121e97a3bd5948
Name: ACS Housing Filtered (TN,AL,GE) | Type: Feature Service | Size: 0.0 MB | ID: 7e6604e9954d4d3ab1398091c72ea5a6
Name: Landscape Conservation Priority scores copy | Type: Feature Service | Size: 0.0 MB | ID: f03db5533e48450f9f30d16b96ab54d6
Name: thrive_region_inverted_poly_2 | Type: Feature Service | Size: 0.37 MB | ID: 509279d977b34c9996de995076c68901
Name: PALD_thrive_region_farmland_trust_2025 | Type: Feature Service | Size: 0.48 MB | ID: c6c00112223c4412a93fed76a8086e

## Find where a dataset is being used.  
If the dataset is not used in any application, it is a zombie dataset. 

In [ ]:

dataname = "NHD_waterbodies_Thrive_region_simplified"
# 1. Target the specific dataset by its exact title and owner
target_layer_items = gis.content.search(query=f'title:"{dataname}" AND owner:{gis.users.me.username} AND type:"Feature Service"', max_items=1)

if not target_layer_items:
    print("Could not find a Feature Service item named 'thrive boundary 2'. Please check the exact spelling.")
    exit()

target_item = target_layer_items[0]
target_url = target_item.url
print(f"Target found: '{target_item.title}' (ID: {target_item.id})")
print(f"Service URL: {target_url}\n")

# 2. Grab all potential containers (maps, apps, dashboards, experiences)
containers = gis.content.search(
    query=f"owner:{gis.users.me.username} AND (type:'Web Map' OR type:'Web Scene' OR type:'Web Experience' OR type:'Dashboard')", 
    max_items=1000
)

print(f"Scanning {len(containers)} maps and applications for dependencies...")
found_in = []

# 3. Scan each item's configuration JSON for the layer's URL or Item ID
for container in containers:
    try:
        data = container.get_data()
        if not data:
            continue
            
        # Convert configuration data to a string for a fast substring match
        data_str = json.dumps(data)
        
        # Check if either the unique service URL or the Item ID is embedded in the application setup
        if (target_url and target_url in data_str) or (target_item.id in data_str):
            found_in.append({
                "Title": container.title,
                "Type": container.type,
                "ID": container.id
            })
    except Exception:
        continue

# 4. Output the dependency list
print("\n--- Usage Results ---")
if found_in:
    print(f"The layer '{target_item.title}' is actively used in the following {len(found_in)} items:")
    for item in found_in:
        print(f"- [{item['Type']}] {item['Title']} (ID: {item['ID']})")
else:
    print(f"The layer '{target_item.title}' is a zombie dataset. It is not referenced in any Web Maps, Scenes, Dashboards, or Experiences in this directory.")

Target found: 'NHD_waterbodies_Thrive_region_simplified' (ID: f979a926ffcb4e4bbfbb673fc679039f)
Service URL: https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/rest/services/NHD_waterbodies_Thrive_region_simplified/FeatureServer



c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#t

Scanning 17 maps and applications for dependencies...


c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cansu\miniconda3\envs\gs-agent\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thrive-geohub.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#t


--- Usage Results ---
The layer 'NHD_waterbodies_Thrive_region_simplified' is a zombie dataset. It is not referenced in any Web Maps, Scenes, Dashboards, or Experiences in this directory.
